# 04 - Dashboard data for Tableau

Tableau reads plain CSVs, so this notebook only **reshapes** what earlier stages produced into a small star schema (one row per store, one row per store-day) and writes it to `data/analytics/`.
No new analysis happens here.

| File | One row per | Used for |
|---|---|---|
| `dim_store.csv` | store | slicers, store type / competition bands, store-level uplift ranking |
| `fact_sales.csv` | store-day | promo vs no-promo comparisons (Stage 1-2 data) |
| `fact_forecast.csv` | store-day, last 6 weeks | forecast vs actual (Stage 3 predictions) |
| `uplift_by_segment.csv` | segment | model-based uplift and extra sales (already written in Stage 3) |

It also prints the **expected value of every dashboard measure**, computed in pandas, so each Tableau number can be checked against a known value.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN, ANALYTICS = Path("../data/clean"), Path("../data/analytics")
sales = pd.read_csv(CLEAN / "sales_clean.csv", parse_dates=["Date"])
pred = pd.read_csv(ANALYTICS / "predictions_test.csv", parse_dates=["Date"])
seg = pd.read_csv(ANALYTICS / "uplift_by_segment.csv")
print(sales.shape, pred.shape, seg.shape)

(38037, 22) (1824, 8) (8, 8)


## dim_store: one row per store

Store-level uplift here uses the same method as Stage 2 (Mon-Fri only, each store's promo days vs its own non-promo days), so the ranking in Tableau matches the notebook.
`DistanceBandOrder` lets Tableau sort the bands Closest -> Farthest instead of alphabetically.

In [2]:
wk = sales[sales.DayOfWeek <= 5]
by_store = wk.groupby(["Store", "Promo"]).Sales.mean().unstack().rename(columns={0: "AvgSalesNoPromo", 1: "AvgSalesPromo"})
by_store["UpliftPct"] = (by_store.AvgSalesPromo / by_store.AvgSalesNoPromo - 1) * 100

dim_store = (sales.groupby("Store")
             .agg(StoreType=("StoreType", "first"), Assortment=("Assortment", "first"),
                  CompetitionDistance=("CompetitionDistance", "first"), Promo2=("Promo2", "first"))
             .join(by_store).reset_index())
dim_store["DistanceBand"] = pd.qcut(dim_store.CompetitionDistance, 4, labels=["Closest", "Near", "Far", "Farthest"]).astype(str)
dim_store["DistanceBandOrder"] = dim_store.DistanceBand.map({"Closest": 1, "Near": 2, "Far": 3, "Farthest": 4})
dim_store["StoreLabel"] = "Store " + dim_store.Store.astype(str) + " (type " + dim_store.StoreType + ")"
dim_store = dim_store.round({"AvgSalesNoPromo": 1, "AvgSalesPromo": 1, "UpliftPct": 2})

assert dim_store.Store.is_unique and len(dim_store) == 50
dim_store.head()

,Store,StoreType,Assortment,CompetitionDistance,Promo2,AvgSalesNoPromo,AvgSalesPromo,UpliftPct,DistanceBand,DistanceBandOrder,StoreLabel
0,5,a,a,29910.0,0,4164.2,6096.7,46.41,Farthest,4,Store 5 (type a)
1,14,a,a,1300.0,1,4997.3,6697.8,34.03,Near,2,Store 14 (type a)
2,55,a,a,720.0,0,3945.9,5757.6,45.91,Near,2,Store 55 (type a)
3,85,b,a,1870.0,0,5958.0,7550.3,26.72,Far,3,Store 85 (type b)
4,96,a,a,8780.0,1,4420.8,7047.6,59.42,Farthest,4,Store 96 (type a)


## fact_sales: one row per store-day

In [3]:
fact_sales = sales[["Store", "Date", "DayOfWeek", "Sales", "Customers", "Promo", "StateHoliday", "SchoolHoliday"]].copy()
fact_sales["PromoLabel"] = np.where(fact_sales.Promo == 1, "Promo", "No promo")  # readable text for slicers and legends

assert not fact_sales.duplicated(["Store", "Date"]).any()
assert set(fact_sales.Store) <= set(dim_store.Store)  # every fact row has a store to relate to
fact_sales.head()

,Store,Date,DayOfWeek,Sales,Customers,Promo,StateHoliday,SchoolHoliday,PromoLabel
0,5,2013-01-02,3,4253,577,0,none,1,No promo
1,5,2013-01-03,4,3465,491,0,none,0,No promo
2,5,2013-01-04,5,4456,533,0,none,0,No promo
3,5,2013-01-05,6,1590,202,0,none,0,No promo
4,5,2013-01-07,1,6978,717,1,none,0,Promo


## fact_forecast: actual vs forecast on the held-out 6 weeks

Error columns are calculated here, row by row, so Tableau calculations can be simple averages and sums. Column names are unique across the tables (`ForecastDate`, `ActualSales`), so Tableau never has to tell apart two fields both called `Sales`.

In [4]:
fact_forecast = pred.rename(columns={"Date": "ForecastDate", "Sales": "ActualSales", "pred_lgbm": "Forecast", "pred_baseline": "BaselineForecast"})[
    ["Store", "ForecastDate", "ActualSales", "Forecast", "BaselineForecast"]].copy()
fact_forecast["Forecast"] = fact_forecast.Forecast.round(1)
fact_forecast["BaselineForecast"] = fact_forecast.BaselineForecast.round(1)
fact_forecast["Error"] = (fact_forecast.Forecast - fact_forecast.ActualSales).round(1)           # forecast minus actual
fact_forecast["AbsPctError"] = ((fact_forecast.Error.abs() / fact_forecast.ActualSales) * 100).round(3)

assert set(fact_forecast.Store) <= set(dim_store.Store)
fact_forecast.head()

,Store,ForecastDate,ActualSales,Forecast,BaselineForecast,Error,AbsPctError
0,5,2015-06-20,2088,1979.4,2080.0,-108.6,5.201
1,14,2015-06-20,3472,3511.0,3526.0,39.0,1.123
2,55,2015-06-20,3475,2698.7,2985.1,-776.3,22.340
3,85,2015-06-20,4615,5420.8,5559.2,805.8,17.460
4,96,2015-06-20,1935,1801.3,1916.4,-133.7,6.910


## Save

In [5]:
dim_store.to_csv(ANALYTICS / "dim_store.csv", index=False)
fact_sales.to_csv(ANALYTICS / "fact_sales.csv", index=False)
fact_forecast.to_csv(ANALYTICS / "fact_forecast.csv", index=False)
for name in ["dim_store", "fact_sales", "fact_forecast", "uplift_by_segment"]:
    f = ANALYTICS / f"{name}.csv"
    print(f"{name:20s} {len(pd.read_csv(f)):6,d} rows  {f.stat().st_size / 1024:7.0f} KB")

dim_store                50 rows        3 KB
fact_sales           38,037 rows     1600 KB
fact_forecast         1,824 rows       85 KB
uplift_by_segment         8 rows        1 KB


## Tableau-ready summary files (drag-and-drop, no formulas)

For a first-time Tableau user, each file is already shaped so that a chart needs no formulas. Uplift, bias and error columns are stored as fractions (0.377 = 37.7%) so Tableau can show them with its normal percentage format.

Charts that should filter each other on a dashboard must come from the **same file**, so the store-level file feeds three sheets (by type, by distance, by store) and the forecast file carries the store type, so a filter can narrow the line chart.

In [6]:
wk = fact_sales[fact_sales.DayOfWeek <= 5]
pm = wk.groupby("Promo").Sales.mean()

# 1) one row of headline numbers, one card each
kpis = pd.DataFrame([{
    "PromoUplift": round(pm[1] / pm[0] - 1, 4),
    "StoresGaining": int((dim_store.UpliftPct > 0).sum()),
    "ExtraSales6wk": round(seg[seg.segment_type == "store_type"].total_extra_sales_holdout.sum()),
    "ForecastMAPE": round(fact_forecast.AbsPctError.mean() / 100, 4),
    "ForecastBias": round(fact_forecast.Forecast.sum() / fact_forecast.ActualSales.sum() - 1, 4),
    "ForecastMAE": round(fact_forecast.Error.abs().mean(), 1),
}])

# 2) promo vs no promo, weekdays only
promo_compare = (wk.assign(spend=wk.Sales / wk.Customers).groupby("PromoLabel")
                 .agg(AvgSales=("Sales", "mean"), AvgCustomers=("Customers", "mean"), SpendPerCustomer=("spend", "mean"))
                 .round(2).reset_index())

# 3) one row per store: feeds the by-type, by-distance and by-store sheets, so clicking a bar can filter the others.
#    The leading number in DistanceGroup makes Tableau's alphabetical sort read Closest -> Farthest.
order = {"Closest": 1, "Near": 2, "Far": 3, "Farthest": 4}
rng = dim_store.groupby("DistanceBand").CompetitionDistance.agg(["min", "max"])
dist_label = {b: f"{o} {b} ({rng.loc[b, 'min']:,.0f}-{rng.loc[b, 'max']:,.0f} m)" for b, o in order.items()}
store_uplift = pd.DataFrame({
    "StoreLabel": dim_store.StoreLabel,
    "TypeGroup": "Type " + dim_store.StoreType,
    "DistanceGroup": dim_store.DistanceBand.map(dist_label),
    "Uplift": (dim_store.UpliftPct / 100).round(4),
})

# 4) forecast vs actual per store in long form (one field colours the two lines; the store type lets a filter narrow the chart)
fb = fact_forecast.merge(dim_store[["Store", "StoreLabel", "StoreType"]], on="Store")
forecast_by_store = (fb.assign(TypeGroup="Type " + fb.StoreType)
                     .melt(id_vars=["StoreLabel", "TypeGroup", "ForecastDate"], value_vars=["ActualSales", "Forecast"], var_name="Series", value_name="Sales")
                     .assign(Series=lambda d: d.Series.map({"ActualSales": "Actual sales", "Forecast": "Forecast"}))
                     .rename(columns={"ForecastDate": "Date"}).round({"Sales": 0}))

# the files must agree with the numbers the notebooks and slides already quote
assert abs(kpis.PromoUplift[0] - 0.377) < 0.001 and kpis.StoresGaining[0] == 50
assert abs(kpis.ForecastMAPE[0] - 0.106) < 0.001 and abs(kpis.ForecastBias[0] + 0.063) < 0.001
by_type = store_uplift.groupby("TypeGroup").Uplift.mean() * 100
by_dist = store_uplift.groupby("DistanceGroup").Uplift.mean() * 100
assert np.allclose(by_type, [40.8, 20.5, 46.1, 38.9], atol=0.06)          # points; tolerance, not exact rounding
assert np.allclose(by_dist, [39.3, 41.6, 38.4, 38.8], atol=0.06)
assert len(store_uplift) == 50 and store_uplift.DistanceGroup.notna().all()
sums = forecast_by_store.groupby("Series").Sales.sum()
assert np.isclose(sums["Actual sales"], fact_forecast.ActualSales.sum(), rtol=1e-4) and np.isclose(sums["Forecast"], fact_forecast.Forecast.sum(), rtol=1e-4)

files = {"tb_kpis": kpis, "tb_promo_compare": promo_compare, "tb_store_uplift": store_uplift, "tb_forecast_by_store": forecast_by_store}
for name, frame in files.items():
    frame.to_csv(ANALYTICS / f"{name}.csv", index=False)
    print(f"{name:22s} {len(frame):5d} rows  columns: {list(frame.columns)}")

tb_kpis                    1 rows  columns: ['PromoUplift', 'StoresGaining', 'ExtraSales6wk', 'ForecastMAPE', 'ForecastBias', 'ForecastMAE']
tb_promo_compare           2 rows  columns: ['PromoLabel', 'AvgSales', 'AvgCustomers', 'SpendPerCustomer']
tb_store_uplift           50 rows  columns: ['StoreLabel', 'TypeGroup', 'DistanceGroup', 'Uplift']
tb_forecast_by_store    3648 rows  columns: ['StoreLabel', 'TypeGroup', 'Date', 'Series', 'Sales']


## Expected values for the Tableau dashboard

Each line is what the matching Tableau calculated field in `docs/tableau_guide.md` should show **with no filters applied**. If a number differs, the relationship or the calculation is wrong, not the data.

In [7]:
wk = fact_sales[fact_sales.DayOfWeek <= 5]
m = wk.groupby("Promo").Sales.mean()
per_day = wk.assign(spend=wk.Sales / wk.Customers).groupby("Promo")[["Customers", "spend"]].mean()
type_uplift = dim_store.groupby("StoreType").UpliftPct.mean()
band_uplift = dim_store.groupby("DistanceBand").UpliftPct.mean()
extra = seg[seg.segment_type == "store_type"].total_extra_sales_holdout.sum()

expected = {
    "Avg Sales Promo (Mon-Fri)": m[1],
    "Avg Sales No Promo (Mon-Fri)": m[0],
    "Promo Uplift % (pooled)": (m[1] / m[0] - 1) * 100,
    "Avg Store Uplift %": dim_store.UpliftPct.mean(),
    "Stores with positive uplift": int((dim_store.UpliftPct > 0).sum()),
    "Customers per day (promo / no promo)": f"{per_day.Customers[1]:,.0f} / {per_day.Customers[0]:,.0f}",
    "Spend per customer (promo / no promo)": f"{per_day.spend[1]:.2f} / {per_day.spend[0]:.2f}",
    "Avg Store Uplift % - type a / b / c / d": " / ".join(f"{type_uplift[t]:.1f}" for t in "abcd"),
    "Avg Store Uplift % - Closest / Near / Far / Farthest": " / ".join(f"{band_uplift[b]:.1f}" for b in ["Closest", "Near", "Far", "Farthest"]),
    "Actual sales, last 6 weeks": fact_forecast.ActualSales.sum(),
    "Forecast sales, last 6 weeks": fact_forecast.Forecast.sum(),
    "Forecast bias %": (fact_forecast.Forecast.sum() / fact_forecast.ActualSales.sum() - 1) * 100,
    "Forecast MAPE %": fact_forecast.AbsPctError.mean(),
    "Forecast MAE (EUR)": fact_forecast.Error.abs().mean(),
    "Extra sales from promos, last 6 weeks (EUR)": extra,
}
for k, v in expected.items():
    print(f"{k:55s} {v:>16,.1f}" if isinstance(v, (float, np.floating)) else f"{k:55s} {str(v):>16s}")

Avg Sales Promo (Mon-Fri)                                        8,242.3
Avg Sales No Promo (Mon-Fri)                                     5,987.4
Promo Uplift % (pooled)                                             37.7
Avg Store Uplift %                                                  39.5
Stores with positive uplift                                           50
Customers per day (promo / no promo)                           864 / 727
Spend per customer (promo / no promo)                       10.07 / 8.76
Avg Store Uplift % - type a / b / c / d                 40.8 / 20.5 / 46.1 / 38.9
Avg Store Uplift % - Closest / Near / Far / Farthest    39.3 / 41.6 / 38.4 / 38.8
Actual sales, last 6 weeks                                      13201465
Forecast sales, last 6 weeks                                12,364,217.5
Forecast bias %                                                     -6.3
Forecast MAPE %                                                     10.6
Forecast MAE (EUR)               